Exercise: _Load the MNIST data and split it into a training set, a validation set, and a test set (e.g., use 50,000 instances for training, 10,000 for validation, and 10,000 for testing)._

In [1]:
from sklearn.datasets import fetch_openml

X_mnist, y_mnist = fetch_openml('mnist_784', return_X_y=True, as_frame=False,
                                parser='auto')

In [2]:
x_train , y_train = X_mnist[:50_000] , y_mnist[:50_000]
x_valid, y_valid = X_mnist[50_000:60_000], y_mnist[50_000:60_000]
x_test, y_test = X_mnist[60_000:], y_mnist[60_000:]

Exercise: _Then train various classifiers, such as a Random Forest classifier, an Extra-Trees classifier, and an SVM._

In [4]:
from sklearn.ensemble import ExtraTreesClassifier ,RandomForestClassifier
from sklearn.svm import LinearSVC
from sklearn.neural_network import MLPClassifier

In [31]:
random_forest_clf = RandomForestClassifier(n_estimators = 100 , random_state = 42)
extra_trees_clf = ExtraTreesClassifier(n_estimators = 100 , random_state = 42)
svm_clf = LinearSVC(max_iter=200, random_state=42,dual=True, tol=0.0001) # changed tol from 20 to 0.0001
mlp_clf = MLPClassifier(random_state = 42)

In [33]:
estimators = [random_forest_clf, extra_trees_clf, svm_clf, mlp_clf]
for estimator in estimators:
    print("Training the", estimator)
    estimator.fit(x_train , y_train )

Training the RandomForestClassifier(random_state=42)
Training the ExtraTreesClassifier(random_state=42)
Training the LinearSVC(dual=True, max_iter=200, random_state=42)


/opt/anaconda3/lib/python3.12/site-packages/sklearn/svm/_base.py:1237: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


Training the MLPClassifier(random_state=42)


In [34]:
[estimator.score(x_valid, y_valid) for estimator in estimators]

[0.9736, 0.9743, 0.8881, 0.966]

Exercise: _Next, try to combine \[the classifiers\] into an ensemble that outperforms them all on the validation set, using a soft or hard voting classifier._

In [35]:
from sklearn.ensemble import VotingClassifier

In [36]:
named_estimators = [
    ("random_forest_clf", random_forest_clf),
    ("extra_trees_clf", extra_trees_clf),
    ("svm_clf", svm_clf),
    ("mlp_clf", mlp_clf),
]

In [37]:
voting_clf = VotingClassifier(named_estimators)

In [38]:
voting_clf.fit(x_train, y_train)

/opt/anaconda3/lib/python3.12/site-packages/sklearn/svm/_base.py:1237: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


VotingClassifier(estimators=[('random_forest_clf',
                              RandomForestClassifier(random_state=42)),
                             ('extra_trees_clf',
                              ExtraTreesClassifier(random_state=42)),
                             ('svm_clf',
                              LinearSVC(dual=True, max_iter=200,
                                        random_state=42)),
                             ('mlp_clf', MLPClassifier(random_state=42))])

In [46]:
voting_clf.score(x_valid, y_valid)

0.9749

In [49]:
voting_clf.estimators

[('random_forest_clf', RandomForestClassifier(random_state=42)),
 ('extra_trees_clf', ExtraTreesClassifier(random_state=42)),
 ('svm_clf', LinearSVC(dual=True, max_iter=200, random_state=42)),
 ('mlp_clf', MLPClassifier(random_state=42))]

In [51]:
voting_clf.estimators_

[RandomForestClassifier(random_state=42),
 ExtraTreesClassifier(random_state=42),
 LinearSVC(dual=True, max_iter=200, random_state=42),
 MLPClassifier(random_state=42)]

In [53]:
voting_clf.named_estimators_

{'random_forest_clf': RandomForestClassifier(random_state=42),
 'extra_trees_clf': ExtraTreesClassifier(random_state=42),
 'svm_clf': LinearSVC(dual=True, max_iter=200, random_state=42),
 'mlp_clf': MLPClassifier(random_state=42)}

The `VotingClassifier` made a clone of each classifier, and it trained the clones using class indices as the labels, not the original class names. Therefore, to evaluate these clones we need to provide class indices as well. To convert the classes to class indices, we can use a `LabelEncoder`:

In [66]:
from sklearn.preprocessing import LabelEncoder

encoder = LabelEncoder()
y_valid_encoded = encoder.fit_transform(y_valid)

In [70]:
import numpy as np
y_valid_encoded = y_valid.astype(np.int64)

In [72]:
[estimator.score(x_valid, y_valid_encoded)
 for estimator in voting_clf.estimators_]

[0.9736, 0.9743, 0.8881, 0.966]

Let's remove the SVM to see if performance improves. It is possible to remove an estimator by setting it to `"drop"` using `set_params()` like this:

In [74]:
voting_clf.set_params(svm_clf="drop")

VotingClassifier(estimators=[('random_forest_clf',
                              RandomForestClassifier(random_state=42)),
                             ('extra_trees_clf',
                              ExtraTreesClassifier(random_state=42)),
                             ('svm_clf', 'drop'),
                             ('mlp_clf', MLPClassifier(random_state=42))])

In [76]:
voting_clf.estimators_

[RandomForestClassifier(random_state=42),
 ExtraTreesClassifier(random_state=42),
 LinearSVC(dual=True, max_iter=200, random_state=42),
 MLPClassifier(random_state=42)]

In [78]:
voting_clf.estimators

[('random_forest_clf', RandomForestClassifier(random_state=42)),
 ('extra_trees_clf', ExtraTreesClassifier(random_state=42)),
 ('svm_clf', 'drop'),
 ('mlp_clf', MLPClassifier(random_state=42))]

So we can either fit the `VotingClassifier` again, or just remove the SVM from the list of trained estimators, both in `estimators_` and `named_estimators_`:

In [83]:
svm_clf_trained = voting_clf.named_estimators_.pop("svm_clf")
voting_clf.estimators_.remove(svm_clf_trained)

In [85]:
voting_clf.score(x_valid, y_valid)

0.9769

In [87]:
voting_clf.voting = "soft"

In [93]:
voting_clf.score(x_valid, y_valid)

0.9769

In [95]:
voting_clf.voting = "hard"
voting_clf.score(x_test, y_test)

0.9727

In [97]:
[estimator.score(x_test, y_test.astype(np.int64))
 for estimator in voting_clf.estimators_]

[0.968, 0.9703, 0.965]